# KOSDAQ 기초통계량

`kosdaq_clean.csv`를 불러와 2010-01-01부터 2025-12-31까지의 수익률, 외국인 순매수비율, 기관 순매수비율, 개인 순매수비율 기초통계량을 확인합니다.

In [ ]:
from pathlib import Path

import pandas as pd

def find_kosdaq_csv() -> Path:
    candidate_paths = [
        Path("..") / "시계열 자료" / "kosdaq_clean.csv",
        Path.cwd().parent / "시계열 자료" / "kosdaq_clean.csv",
        Path("시계열 자료") / "kosdaq_clean.csv",
        Path.cwd() / "시계열 자료" / "kosdaq_clean.csv",
    ]
    for path in candidate_paths:
        resolved = path.resolve()
        if resolved.exists():
            return resolved

    matches = sorted(Path.cwd().rglob("kosdaq_clean.csv"))
    if not matches:
        raise FileNotFoundError("kosdaq_clean.csv 파일을 찾지 못했습니다.")
    return matches[0].resolve()


data_path = find_kosdaq_csv()

kosdaq = pd.read_csv(data_path, parse_dates=["date"])
kosdaq = kosdaq[(kosdaq["date"] >= "2010-01-01") & (kosdaq["date"] <= "2025-12-31")].copy()
print(f"Loaded: {data_path}")
print(f"Period: {kosdaq['date'].min().date()} ~ {kosdaq['date'].max().date()} / rows={len(kosdaq):,}")
kosdaq.head()

In [ ]:
stats_cols = {
    "return_pct": "수익률",
    "foreign_net_buy_ratio": "외국인 순매수비율",
    "inst_net_buy_ratio": "기관 순매수비율",
    "personal_net_buy_ratio": "개인 순매수비율",
}

from scipy.stats import ttest_1samp

selected = kosdaq[list(stats_cols)]
t_stats = selected.apply(lambda col: ttest_1samp(col.dropna(), popmean=0.0).statistic)
p_values = selected.apply(lambda col: ttest_1samp(col.dropna(), popmean=0.0).pvalue)

basic_stats = pd.DataFrame({
    "mean": selected.mean(),
    "t-value": t_stats,
    "p-value": p_values,
    "std": selected.std(),
    "min": selected.min(),
    "max": selected.max(),
    "kurtosis": selected.kurt(),
    "skewness": selected.skew(),
})
basic_stats.index = [stats_cols[col] for col in basic_stats.index]
basic_stats

In [ ]:
from IPython.display import display

named_selected = selected.rename(columns=stats_cols)
cov_matrix = named_selected.cov()
corr_matrix = named_selected.corr()

print("Covariance Matrix")
display(cov_matrix)
print("Correlation Matrix")
display(corr_matrix)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import font_manager, rcParams

preferred_fonts = ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR", "Noto Sans KR"]
available_fonts = {font.name for font in font_manager.fontManager.ttflist}
selected_font = None
for font_name in preferred_fonts:
    if font_name in available_fonts:
        selected_font = font_name
        break

sns.set_theme(style="whitegrid")
if selected_font is not None:
    rcParams["font.family"] = "sans-serif"
    rcParams["font.sans-serif"] = [selected_font]
rcParams["axes.unicode_minus"] = False
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

return_series = selected["return_pct"].dropna()
sns.histplot(return_series, bins=40, kde=True, ax=axes[0], color="#4C72B0", edgecolor="white")
axes[0].axvline(0, color="#D62728", linestyle="--", linewidth=1.2, label="0")
axes[0].axvline(return_series.mean(), color="#2CA02C", linestyle=":", linewidth=1.5, label="평균")
axes[0].set_title("수익률 분포", fontsize=13)
axes[0].set_xlabel("수익률")
axes[0].set_ylabel("빈도")
axes[0].legend()

flow_cols = ["foreign_net_buy_ratio", "inst_net_buy_ratio", "personal_net_buy_ratio"]
flow_colors = {
    "foreign_net_buy_ratio": "#1f77b4",
    "inst_net_buy_ratio": "#ff7f0e",
    "personal_net_buy_ratio": "#2ca02c",
}

for col in flow_cols:
    series = selected[col].dropna()
    sns.kdeplot(series, ax=axes[1], label=stats_cols[col], color=flow_colors[col], linewidth=2)
    axes[1].axvline(series.mean(), color=flow_colors[col], linestyle=":", linewidth=1.2)

axes[1].axvline(0, color="#D62728", linestyle="--", linewidth=1.2, label="0")
axes[1].set_title("순매수비율 비교 분포", fontsize=13)
axes[1].set_xlabel("순매수비율")
axes[1].set_ylabel("밀도")
axes[1].legend(title="변수")

fig.suptitle("KOSDAQ 수익률 및 수급비율 분포", fontsize=16, y=1.02)
fig.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()
